# StyleMatch Source Import

Imports manually collected original-language `.txt` files from a manifest into the corpus, chunks them, and audits coverage.

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/stylematch_v1')
REPO = Path('/content/style_matching')
assert REPO.exists(), 'Clone the repo to /content/style_matching first.'
%cd /content/style_matching

In [ ]:
!git pull
!git rev-parse --short HEAD
!python scripts/audit_source_registry.py

In [ ]:
drive_manifest = DRIVE_ROOT / 'data/source_registry/source_manifest.csv'
repo_manifest = REPO / 'data/source_registry/source_manifest.csv'
MANIFEST = drive_manifest if drive_manifest.exists() else repo_manifest
assert MANIFEST.exists(), f'Missing manifest in Drive or repo: {drive_manifest} | {repo_manifest}'
MANIFEST

In [ ]:
!python scripts/import_source_manifest.py "{MANIFEST}" --dry-run
!python scripts/import_source_manifest.py "{MANIFEST}" --append
!python scripts/chunk_corpus.py --corpus both --min-words 75 --max-words 150
!python scripts/audit_corpus_coverage.py --corpus both --output data/coverage_audit.json

In [ ]:
import shutil

for rel in [
    'data/literary/meta/sources.csv',
    'data/literary/meta/chunks.csv',
    'data/rhetorical/meta/sources.csv',
    'data/rhetorical/meta/chunks.csv',
    'data/coverage_audit.json',
]:
    src = REPO / rel
    if src.exists():
        dst = DRIVE_ROOT / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        print('copied', rel)